# RAG Evaluation (Không cần Ground Truth)

Notebook đánh giá hệ thống RAG theo hướng reference-free (RAGAS-style):
- Context Relevance
- Answer Relevance
- Faithfulness
- Latency, Error Rate


## 1) Cài thư viện

In [ ]:
# %pip install requests pandas matplotlib

## 2) Cấu hình

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import json
import re
import statistics
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Backend cần đánh giá
BASE_URL = "http://localhost:8000"
SEARCH_ENDPOINT = "/api/v1/search"
CHAT_ENDPOINT = "/api/v1/chat"

# Dùng tập query hiện có (không dùng relevant_chunk_ids)
DATASET_PATH = Path("../dataset/query_set.v1.json")

# Judge model (OpenAI-compatible)
JUDGE_BASE_URL = os.getenv("JUDGE_BASE_URL", "https://api.openai.com/v1")
JUDGE_API_KEY = os.getenv("JUDGE_API_KEY", "")
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "gpt-4o-mini")

TOP_K_CONTEXT = 5
REQUEST_TIMEOUT_SEC = 30

OUTPUT_ROOT = Path("../output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
out_dir = OUTPUT_ROOT / run_id
charts_dir = out_dir / "charts"
out_dir.mkdir(parents=True, exist_ok=True)
charts_dir.mkdir(parents=True, exist_ok=True)

assert DATASET_PATH.exists(), f"Không tìm thấy file query: {DATASET_PATH.resolve()}"
assert JUDGE_API_KEY, "Thiếu JUDGE_API_KEY trong môi trường"

print("Output dir:", out_dir.resolve())

## 3) Hàm gọi API

In [ ]:
def call_search(query, university_code=None):
    payload = {"query": query}
    if university_code:
        payload["university_code"] = university_code

    url = BASE_URL.rstrip("/") + SEARCH_ENDPOINT
    t0 = time.perf_counter()
    res = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT_SEC)
    latency_ms = (time.perf_counter() - t0) * 1000
    res.raise_for_status()
    body = res.json()
    hits = body.get("hits", []) if isinstance(body, dict) else []
    return hits, latency_ms

def call_chat(query, university_code=None):
    payload = {"query": query}
    if university_code:
        payload["university_code"] = university_code

    url = BASE_URL.rstrip("/") + CHAT_ENDPOINT
    t0 = time.perf_counter()
    res = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT_SEC)
    latency_ms = (time.perf_counter() - t0) * 1000
    res.raise_for_status()
    body = res.json()
    answer = body.get("answer", "") if isinstance(body, dict) else ""
    used_chunks = body.get("used_chunks", 0) if isinstance(body, dict) else 0
    return answer, used_chunks, latency_ms


## 4) Hàm chấm điểm Judge (reference-free)

In [ ]:
def extract_json(text):
    text = text.strip()
    # Bóc JSON trong code fence nếu có
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if m:
        text = m.group(1)
    return json.loads(text)

def judge_rag(query, contexts, answer):
    prompt = {
        "query": query,
        "contexts": contexts,
        "answer": answer,
        "instruction": (
            "Hãy chấm 3 điểm từ 0 đến 1 cho hệ RAG và trả JSON thuần với đúng các khóa: "
            "context_relevance, answer_relevance, faithfulness, explanation. "
            "- context_relevance: context có liên quan truy vấn đến mức nào. "
            "- answer_relevance: câu trả lời có trả lời đúng câu hỏi không. "
            "- faithfulness: câu trả lời có bám context, không bịa thêm không."
        )
    }

    url = JUDGE_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {
        "Authorization": f"Bearer {JUDGE_API_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": JUDGE_MODEL,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": "Bạn là giám khảo đánh giá RAG khách quan."},
            {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)},
        ],
    }

    res = requests.post(url, headers=headers, json=body, timeout=REQUEST_TIMEOUT_SEC)
    res.raise_for_status()
    out = res.json()
    content = out["choices"][0]["message"]["content"]
    parsed = extract_json(content)

    for k in ["context_relevance", "answer_relevance", "faithfulness"]:
        parsed[k] = float(parsed.get(k, 0.0))
        if parsed[k] < 0:
            parsed[k] = 0.0
        if parsed[k] > 1:
            parsed[k] = 1.0

    parsed["explanation"] = str(parsed.get("explanation", ""))
    return parsed


## 5) Chạy đánh giá

In [ ]:
dataset = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
rows = []

for i, item in enumerate(dataset, start=1):
    query_id = str(item.get("id", i))
    query = str(item.get("query", "")).strip()
    university_code = str(item.get("university_code", "") or "")
    intent = str(item.get("intent", "") or "")

    status = "ok"
    error = ""
    search_latency_ms = 0.0
    chat_latency_ms = 0.0
    judge_latency_ms = 0.0
    contexts = []
    answer = ""
    used_chunks = 0
    context_relevance = 0.0
    answer_relevance = 0.0
    faithfulness = 0.0
    explanation = ""

    try:
        hits, search_latency_ms = call_search(query, university_code if university_code else None)
        contexts = [str(h.get("text", ""))[:1200] for h in hits[:TOP_K_CONTEXT]]

        answer, used_chunks, chat_latency_ms = call_chat(query, university_code if university_code else None)

        t0_judge = time.perf_counter()
        judge = judge_rag(query, contexts, answer)
        judge_latency_ms = (time.perf_counter() - t0_judge) * 1000

        context_relevance = judge["context_relevance"]
        answer_relevance = judge["answer_relevance"]
        faithfulness = judge["faithfulness"]
        explanation = judge["explanation"]
    except Exception as ex:
        status = "error"
        error = str(ex)

    rows.append({
        "id": query_id,
        "query": query,
        "university_code": university_code,
        "intent": intent,
        "status": status,
        "error": error,
        "search_latency_ms": search_latency_ms,
        "chat_latency_ms": chat_latency_ms,
        "judge_latency_ms": judge_latency_ms,
        "total_latency_ms": search_latency_ms + chat_latency_ms + judge_latency_ms,
        "retrieved_context_count": len(contexts),
        "used_chunks": used_chunks,
        "context_relevance": context_relevance,
        "answer_relevance": answer_relevance,
        "faithfulness": faithfulness,
        "rag_score": (context_relevance + answer_relevance + faithfulness) / 3.0,
        "judge_explanation": explanation,
        "answer": answer,
        "contexts": contexts,
    })

    if i % 10 == 0:
        print(f"Đã xử lý {i}/{len(dataset)}")

df = pd.DataFrame(rows)
df.head()

## 6) Tổng hợp metric

In [ ]:
ok_df = df[df["status"] == "ok"].copy()
error_rate = 1.0 - (len(ok_df) / len(df) if len(df) else 0.0)

def safe_mean(series):
    return float(series.mean()) if len(series) else 0.0

def safe_p(series, q):
    return float(series.quantile(q)) if len(series) else 0.0

summary = {
    "query_count_total": int(len(df)),
    "query_count_ok": int(len(ok_df)),
    "query_count_error": int((df["status"] == "error").sum()),
    "error_rate": error_rate,
    "context_relevance_mean": safe_mean(ok_df["context_relevance"]),
    "answer_relevance_mean": safe_mean(ok_df["answer_relevance"]),
    "faithfulness_mean": safe_mean(ok_df["faithfulness"]),
    "rag_score_mean": safe_mean(ok_df["rag_score"]),
    "total_latency_mean_ms": safe_mean(df["total_latency_ms"]),
    "total_latency_p50_ms": safe_p(df["total_latency_ms"], 0.50),
    "total_latency_p95_ms": safe_p(df["total_latency_ms"], 0.95),
    "total_latency_p99_ms": safe_p(df["total_latency_ms"], 0.99),
}

df_summary = pd.DataFrame([
    {"metric": k, "value": v}
    for k, v in summary.items()
]).sort_values("metric")

df_summary

## 7) Biểu đồ

In [ ]:
score_items = {
    "Context Relevance": summary["context_relevance_mean"],
    "Answer Relevance": summary["answer_relevance_mean"],
    "Faithfulness": summary["faithfulness_mean"],
    "RAG Score": summary["rag_score_mean"],
}

plt.figure(figsize=(9, 5))
plt.bar(score_items.keys(), score_items.values())
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Reference-Free RAG Quality Scores")
plt.tight_layout()
plt.savefig(charts_dir / "quality_scores.png", dpi=150)
plt.show()

In [ ]:
lat_items = {
    "Mean": summary["total_latency_mean_ms"],
    "P50": summary["total_latency_p50_ms"],
    "P95": summary["total_latency_p95_ms"],
    "P99": summary["total_latency_p99_ms"],
}

plt.figure(figsize=(8, 5))
plt.bar(lat_items.keys(), lat_items.values())
plt.ylabel("Milliseconds")
plt.title("Latency Distribution")
plt.tight_layout()
plt.savefig(charts_dir / "latency.png", dpi=150)
plt.show()

## 8) Xuất báo cáo

In [ ]:
config = {
    "base_url": BASE_URL,
    "search_endpoint": SEARCH_ENDPOINT,
    "chat_endpoint": CHAT_ENDPOINT,
    "dataset_path": str(DATASET_PATH),
    "judge_base_url": JUDGE_BASE_URL,
    "judge_model": JUDGE_MODEL,
    "top_k_context": TOP_K_CONTEXT,
    "request_timeout_sec": REQUEST_TIMEOUT_SEC,
    "executed_at": datetime.now().isoformat(),
}

(out_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
(out_dir / "raw_results.json").write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
df.to_csv(out_dir / "per_query_metrics.csv", index=False)
df_summary.to_csv(out_dir / "summary_metrics.csv", index=False)

report_lines = [
    "# RAG Evaluation Report (No Ground Truth)",
    "",
    "## Experiment Setup",
    f"- Base URL: {BASE_URL}",
    f"- Search endpoint: {SEARCH_ENDPOINT}",
    f"- Chat endpoint: {CHAT_ENDPOINT}",
    f"- Dataset: {DATASET_PATH}",
    f"- Judge model: {JUDGE_MODEL}",
    f"- Query count: {summary['query_count_total']}",
    f"- Success count: {summary['query_count_ok']}",
    f"- Error count: {summary['query_count_error']}",
    f"- Error rate: {summary['error_rate']:.4f}",
    "",
    "## Reference-Free Quality Metrics",
    f"- Context Relevance: {summary['context_relevance_mean']:.4f}",
    f"- Answer Relevance: {summary['answer_relevance_mean']:.4f}",
    f"- Faithfulness: {summary['faithfulness_mean']:.4f}",
    f"- RAG Score (mean): {summary['rag_score_mean']:.4f}",
    "",
    "## Latency",
    f"- Mean: {summary['total_latency_mean_ms']:.2f} ms",
    f"- P50: {summary['total_latency_p50_ms']:.2f} ms",
    f"- P95: {summary['total_latency_p95_ms']:.2f} ms",
    f"- P99: {summary['total_latency_p99_ms']:.2f} ms",
    "",
    "## Artifacts",
    "- per_query_metrics.csv",
    "- summary_metrics.csv",
    "- raw_results.json",
    "- charts/quality_scores.png",
    "- charts/latency.png",
]

(out_dir / "report.md").write_text("\n".join(report_lines), encoding="utf-8")

print("Đã xuất đầy đủ artifact vào:", out_dir.resolve())